In [27]:
import numpy as np
import pandas as pd
import openmatrix as omx
from pathlib import Path
from dbfread import DBF
import copy
from datetime import datetime
import os


In [28]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


## Functions


In [29]:
def load_block_coefficients(file_path):
    # Initialize with a 'global' bucket for non-segmented variables
    coeffs = {"global": {}}

    with open(file_path, "r") as f:
        for line in f:
            # Strip out comments and whitespace
            line = line.split(";")[0].strip()
            if not line or "=" not in line:
                continue

            name, val = line.split("=")
            name = name.strip().lower()
            val = float(val.strip())

            # 1. Check if it is a segment-specific variable
            if name.endswith("_lo") or name.endswith("_hi"):
                parts = name.split("_")
                seg = f"{parts[-2]}_{parts[-1]}"  # e.g., 0veh_lo
                param = "_".join(parts[:-2])  # e.g., intra, logsum

                if seg not in coeffs:
                    coeffs[seg] = {}
                coeffs[seg][param] = val

            # 2. Otherwise, treat it as a global variable (like short_trip_factor)
            else:
                coeffs["global"][name] = val

    return coeffs


In [30]:
# function used to apply range-based masking for external zones
def apply_range_mask(mask_array, range_str):
    for r in range_str.split(","):
        if "-" in r:
            start, end = map(int, r.split("-"))
            # This correctly maps TAZ 3563 to Index 3562
            mask_array[start - 1 : end] = True
        else:
            mask_array[int(r) - 1] = True


In [31]:
# function to sum values in the household productions creation script
def sum_columns(hbw_productions, prefix_list, veh_suffixes):
    cols = []
    for prefix in prefix_list:
        for v in veh_suffixes:
            cols.append(f"P_{prefix}{v}")
    return hbw_productions[cols].sum(axis=1)


In [32]:
# calculate size term for utility equation (size of destination in terms of nubmer of opportunities)
def get_size_term(seg_coeffs, df_se_input):
    # total employment
    total_emp = df_se_input["TOTEMP"].values.astype(float).copy()

    # precision-safe replacement logic
    epsilon = 1e-6
    for emp_type, coeff_key in [
        ("RETEMP", "retail_emp"),
        ("INDEMP", "industrial_emp"),
        ("OTHEMP", "other_emp"),
    ]:
        coeff = seg_coeffs.get(coeff_key, 1.0)

        if abs(coeff - 1.0) > epsilon:
            raw_emp = df_se_input[emp_type].values
            total_emp += raw_emp * (coeff - 1.0)

    # Ignore the divide by zero warning, let Python evaluate log(0) as -inf
    with np.errstate(divide="ignore", invalid="ignore"):
        ln_size = np.log(total_emp)

    return ln_size


In [33]:
def export_run_for_dashboard(trips_data, df_se_full, run_name, file_name):
    """Rolls up the TAZ matrices to Large Districts for the dashboard CSV,
    AND saves the raw TAZxTAZ matrices to an OMX file for deep-dive metrics."""
    print(f"\n  -> Exporting '{run_name}'...")

    os.makedirs("intermediate/model_runs", exist_ok=True)

    # ---------------------------------------------------------
    # 1. SAVE RAW TAZ MATRICES (For your advanced metrics)
    # ---------------------------------------------------------
    # omx_path = f"intermediate/model_runs/{file_name}.omx"
    #
    ## Open OMX file in write mode ('w' will overwrite if it already exists)
    # with omx.open_file(omx_path, "w") as f:
    #    for seg, data in trips_data.items():
    #        # data["trips"] is the massive 3629x3629 numpy array
    #        f[seg] = data["trips"]
    #
    # print(f"  -> Raw TAZ matrices saved securely to {omx_path}")

    # ---------------------------------------------------------
    # 2. SAVE DISTRICT AGGREGATION (For your facet chart)
    # ---------------------------------------------------------
    dist_lrg_array = df_se_full["DISTLRG"].values
    dist_records = []

    for seg, data in trips_data.items():
        # Melt the TAZ matrix
        df_taz = pd.DataFrame(data["trips"])
        df_taz["p_DistLrg"] = dist_lrg_array
        df_long = df_taz.melt(
            id_vars=["p_DistLrg"], var_name="a_taz", value_name="total_trips"
        )

        # Map a_taz index to a_DistLrg and group
        df_long["a_DistLrg"] = df_long["a_taz"].map(lambda x: dist_lrg_array[x])
        df_dist = df_long.groupby(["p_DistLrg", "a_DistLrg"], as_index=False)[
            "total_trips"
        ].sum()

        # Add identifiers
        df_dist["veh_inc"] = seg
        df_dist["Source"] = run_name
        dist_records.append(df_dist)

    df_final = pd.concat(dist_records, ignore_index=True)

    # Save to disk
    csv_path = f"intermediate/model_runs/{file_name}.csv"
    df_final.to_csv(csv_path, index=False)
    print(f"  -> Dashboard CSV saved successfully to {csv_path}!")


# File Paths & Inputs


In [34]:
path_inputs = Path.cwd() / "inputs"
path_outputs = Path.cwd() / "outputs"
path_start_files = path_inputs / "input_files/C33b"

# path_coeff_file = path_outputs / "coefficients_cal8.block"             # Test 0
path_coeff_file = path_outputs / "coefficients_cal8.block"  # Test 0-33b
# path_coeff_file = path_outputs / "coefficients_cal8_estimated1.block"  # Test 3
# path_coeff_file = path_outputs / "coefficients_cal8_distance1.block"   # Test 4
# path_coeff_file = path_outputs / "coefficients_cal8_distance1a.block"  # Test 4a
# path_coeff_file = path_outputs / "coefficients_cal8_estimated2.block"  # Test 5
# path_coeff_file = path_outputs / "coefficients_cal8_distance2.block"   # Test 6
# path_coeff_file = path_outputs / "coefficients_cal8_estimated3.block"  # Test 7
# path_coeff_file = path_outputs / "coefficients_cal8_distance3.block"   # Test 8
# path_coeff_file = path_outputs / "coefficients_cal8_adjusted1.block"   # Test 9
# path_coeff_file = path_outputs / "coefficients_cal8_adjusted2.block"  # Test 10

path_se_file = path_start_files / "SE_File.dbf"
path_pa_file = path_start_files / "pa_initial.dbf"
path_tel_file = path_start_files / "telecommute.dbf"
path_hh_files = [
    path_start_files / f"HH{i}_PercTrips_segment_hbw.dbf" for i in range(1, 7)
]
path_taz_file = path_start_files / "WFv1000_TAZ.dbf"

# Skims and Logsums (assuming they are already converted to OMX in this folder)
path_skm_hwy = path_start_files / "skm_auto_Pk.omx"
path_skm_walk = path_start_files / "Best_Walk_Skims.omx"
path_logsums = path_start_files / "HBW_logsums_Pk.omx"

# We load this once here just to get the segment names for loading matrices
initial_coeffs = load_block_coefficients(path_coeff_file)
segments = list(initial_coeffs.keys())
segments = list(initial_coeffs.keys())
if "global" in segments:
    segments.remove("global")


In [35]:
used_zones = 3629
dummy_zones_str = "3563-3600"
external_zones_str = "3601-3629"


In [36]:
# read dbf files
df_se = pd.DataFrame(iter(DBF(path_se_file)))
df_pa = pd.DataFrame(iter(DBF(path_pa_file)))
df_tel = pd.DataFrame(iter(DBF(path_tel_file)))
df_hh = [pd.DataFrame(iter(DBF(f))) for f in path_hh_files]
df_taz = pd.DataFrame(iter(DBF(path_taz_file)))

# Create dummy/external mask
mask_external_dummys = np.zeros(used_zones, dtype=bool)
apply_range_mask(mask_external_dummys, dummy_zones_str)
apply_range_mask(mask_external_dummys, external_zones_str)


In [37]:
hts_hh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.hh"
).to_dataframe()
hts_person_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.person"
).to_dataframe()
hts_trip_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.trip_linked"
).to_dataframe()
hts_veh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.vehicle"
).to_dataframe()


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


# Load Static Data


## SE Data, Employment Ratios, District Math


In [ ]:
if "N" in df_se.columns:
    df_se_indexed = df_se.set_index(df_se["N"] - 1)
else:
    df_se_indexed = df_se.copy()

df_se_full = df_se_indexed.reindex(np.arange(used_zones)).fillna(0)
if "N" in df_se_full.columns:
    df_se_full["N"] = np.arange(1, used_zones + 1)

# ==========================================================
# DEFINE LEVEL 2 AND LEVEL 3 AGGREGATE GROUPS
# ==========================================================
# Level 2: Trip Attraction
df_se_full["RETEMP2"] = df_se_full["RETL"] + df_se_full["FOOD"]
df_se_full["INDEMP2"] = df_se_full["MANU"] + df_se_full["WSLE"]
df_se_full["OFFEMP2"] = df_se_full["OFFI"] + df_se_full["GVED"] + df_se_full["HLTH"]
df_se_full["OTHEMP2"] = df_se_full["OTHR"]

# Level 3: Compensation
df_se_full["LOWEMP3"] = df_se_full["RETL"] + df_se_full["FOOD"]
df_se_full["MIDEMP3"] = (
    df_se_full["MANU"] + df_se_full["WSLE"] + df_se_full["GVED"] + df_se_full["OTHR"]
)
df_se_full["HIGHEMP3"] = df_se_full["OFFI"] + df_se_full["HLTH"]


# ==========================================================
# CALCULATE EMPLOYMENT RATIOS
# ==========================================================

# --- Original Level 1 Ratios ---
denom = (df_se_full["RETEMP"] + df_se_full["INDEMP"] + df_se_full["OTHEMP"]).values
retail_ratio = np.divide(
    df_se_full["RETEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
ind_ratio = np.divide(
    df_se_full["INDEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
other_ratio = np.divide(
    df_se_full["OTHEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)

# --- Level 2 Ratios (Trip Attraction) ---
denom_2 = (
    df_se_full["RETEMP2"]
    + df_se_full["INDEMP2"]
    + df_se_full["OFFEMP2"]
    + df_se_full["OTHEMP2"]
).values
ret_ratio2 = np.divide(
    df_se_full["RETEMP2"].values,
    denom_2,
    out=np.zeros_like(denom_2, dtype=float),
    where=denom_2 != 0,
)
ind_ratio2 = np.divide(
    df_se_full["INDEMP2"].values,
    denom_2,
    out=np.zeros_like(denom_2, dtype=float),
    where=denom_2 != 0,
)
off_ratio2 = np.divide(
    df_se_full["OFFEMP2"].values,
    denom_2,
    out=np.zeros_like(denom_2, dtype=float),
    where=denom_2 != 0,
)
oth_ratio2 = np.divide(
    df_se_full["OTHEMP2"].values,
    denom_2,
    out=np.zeros_like(denom_2, dtype=float),
    where=denom_2 != 0,
)

# --- Level 3 Ratios (Compensation) ---
denom_3 = (
    df_se_full["LOWEMP3"] + df_se_full["MIDEMP3"] + df_se_full["HIGHEMP3"]
).values
low_ratio3 = np.divide(
    df_se_full["LOWEMP3"].values,
    denom_3,
    out=np.zeros_like(denom_3, dtype=float),
    where=denom_3 != 0,
)
mid_ratio3 = np.divide(
    df_se_full["MIDEMP3"].values,
    denom_3,
    out=np.zeros_like(denom_3, dtype=float),
    where=denom_3 != 0,
)
high_ratio3 = np.divide(
    df_se_full["HIGHEMP3"].values,
    denom_3,
    out=np.zeros_like(denom_3, dtype=float),
    where=denom_3 != 0,
)

# ==========================================================
# DISTRICT MATCH
# ==========================================================
# District Match boolean matrix
dist_lrg = df_se_full["DISTLRG"].values
district_match = dist_lrg[:, np.newaxis] == dist_lrg[np.newaxis, :]


In [39]:
dist2_factor = initial_coeffs["global"]["beta_dist2"]
dist3_factor = initial_coeffs["global"]["beta_dist3"]
emp_factor = initial_coeffs["global"]["beta_emp"]


## Attractions and Productions


In [40]:
# Prepare Target Attractions
target_attractions = np.zeros(used_zones)
df_pa["Zone_Index"] = df_pa["Z"].astype(int) - 1
valid_pa = df_pa[df_pa["Zone_Index"] < used_zones]
target_attractions[valid_pa["Zone_Index"].values] = valid_pa["HBW_A"].values

# Prepare Trip Productions by Segment
zone_col = "Z"
hh_all = df_hh[0].copy()
for df in df_hh[1:]:
    hh_all = hh_all.merge(df, on=zone_col, suffixes=("", "_dup"))
    for col in df.columns:
        if col != zone_col and col + "_dup" in hh_all.columns:
            hh_all[col] += hh_all[col + "_dup"]
            hh_all.drop(columns=[col + "_dup"], inplace=True)
hbw_prods = hh_all.merge(df_pa[[zone_col, "HBW_P"]], on=zone_col)

# Calculate trips by segment
low_inc, high_inc = ["ILW0", "ILW1", "ILW2", "ILW3"], ["IHW0", "IHW1", "IHW2", "IHW3"]
hbw_prods["trips_0veh_lo"] = (
    sum_columns(hbw_prods, low_inc, ["V0"]) * hbw_prods["HBW_P"]
)
hbw_prods["trips_1veh_lo"] = (
    sum_columns(hbw_prods, low_inc, ["V1"]) * hbw_prods["HBW_P"]
)
hbw_prods["trips_2veh_lo"] = (
    sum_columns(hbw_prods, low_inc, ["V2", "V3"]) * hbw_prods["HBW_P"]
)
hbw_prods["trips_0veh_hi"] = (
    sum_columns(hbw_prods, high_inc, ["V0"]) * hbw_prods["HBW_P"]
)
hbw_prods["trips_1veh_hi"] = (
    sum_columns(hbw_prods, high_inc, ["V1"]) * hbw_prods["HBW_P"]
)
hbw_prods["trips_2veh_hi"] = (
    sum_columns(hbw_prods, high_inc, ["V2", "V3"]) * hbw_prods["HBW_P"]
)

# Zero out external/dummy zones
cols_to_zero = [
    "trips_0veh_lo",
    "trips_1veh_lo",
    "trips_2veh_lo",
    "trips_0veh_hi",
    "trips_1veh_hi",
    "trips_2veh_hi",
]


In [50]:
# Telecommute Factors
pct_tel_hbw = df_tel["PCTTELHBW"].values.reshape(1, -1)
fac_tel_hbw = df_tel["FACTELHBW"].values.reshape(1, -1)

# parking
prk_temp = df_taz["PRKCSTTEMP"].values.reshape(1, -1)
prk_perm = df_taz["PRKCSTPERM"].values.reshape(1, -1)


## Matrix Data


In [42]:
with (
    omx.open_file(path_skm_hwy, "r") as skm_hwy,
    omx.open_file(path_skm_walk, "r") as skm_walk,
):
    dist_mtx = np.array(skm_hwy["dist_GP"][:])
    hwy_time = np.array(skm_hwy["ivt_GP"][:]) + np.array(skm_hwy["ovt"][:])
    walk_gencost = np.array(skm_walk["GENCOST"][:])

log_file = {}
with omx.open_file(path_logsums, "r") as f:
    for seg in segments:
        log_file[seg] = np.array(f[seg][:])

# The heavy lifting: pre-calculate distance variables once!
dist_sq = dist_mtx * dist_mtx
dist_cu = dist_sq * dist_mtx
short_trip_factor = initial_coeffs["global"]["short_trip_factor"] / np.clip(
    dist_mtx, 1.0, None
)


## Observed Trips by VehOwn & Income


In [43]:
hts_trip_23_merge = hts_trip_23.copy()
hts_trip_23_merge = hts_trip_23_merge[
    [
        "unique_id",
        "hh_id",
        "person_id",
        "vehicle_id",
        "pCO_TAZID_USTMv4",
        "aCO_TAZID_USTMv4",
        "pSUBAREAID",
        "aSUBAREAID",
        "PURP7_t",
        "trip_weight",
        "distance_miles",
    ]
]

# filter to HBW
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="pCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "pTAZID"}
)
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="aCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "aTAZID"}
)

# fitler to WF subarea
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["pSUBAREAID"] == 1]
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["aSUBAREAID"] == 1]

# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_23.copy()
    .groupby("hh_id")["vehicle_id"]
    .count()
    .reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["hh_id", "vehown"]]

# income lookup table
income_lookup = hts_hh_23.copy()[["hh_id", "income_detailed"]]
income_lookup["income"] = np.select(
    [
        income_lookup["income_detailed"].isin([1, 2, 3, 4]),
        income_lookup["income_detailed"].isin([5, 6, 7, 8, 9, 10]),
    ],
    ["lo", "hi"],
    default="",
)

income_lookup = income_lookup[["hh_id", "income"]]

# merge vehown and income to trip table
hts_trip_23_merge = hts_trip_23_merge.merge(vehown_lookup, how="left", on="hh_id")
hts_trip_23_merge["vehown"] = hts_trip_23_merge["vehown"].fillna(0).astype(int)
hts_trip_23_merge = hts_trip_23_merge.merge(income_lookup, how="left", on="hh_id")

# calculate segment
hts_trip_23_merge["segment"] = np.where(
    hts_trip_23_merge["income"].notna() & (hts_trip_23_merge["income"] != ""),
    hts_trip_23_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_23_merge["income"].astype(str),
    "",
)

# filter to only known income trips
hts_trip_23_merge = hts_trip_23_merge[~(hts_trip_23_merge["segment"] == "")]


In [44]:
# start final table
df_obs_vehown_inc = hts_trip_23_merge.copy()
df_obs_vehown_inc = df_obs_vehown_inc[["pTAZID", "aTAZID", "segment", "trip_weight"]]
df_obs_vehown_inc = df_obs_vehown_inc.rename(columns={"pTAZID": "i", "aTAZID": "j"})

# Define the 6 segments
segments = ["0veh_lo", "0veh_hi", "1veh_lo", "1veh_hi", "2veh_lo", "2veh_hi"]

# Pivot table
df_obs_pivot = df_obs_vehown_inc.pivot_table(
    index=["i", "j"],
    columns="segment",
    values="trip_weight",
    aggfunc="sum",
    fill_value=0,
)

# Make sure all 6 segment columns exist
for seg in segments:
    if seg not in df_obs_pivot.columns:
        df_obs_pivot[seg] = 0

# Reorder columns
df_obs_pivot = df_obs_pivot[segments]

# Add TOTAL column
df_obs_pivot["TOTAL"] = df_obs_pivot.sum(axis=1)

# Reset index if you want a flat dataframe
df_obs_pivot = df_obs_pivot.reset_index()


## Package Data


In [51]:
static_data = {
    "segments": segments,
    "dist_mtx": dist_mtx,
    "dist_sq": dist_sq,
    "dist_cu": dist_cu,
    "dist2_factor": dist2_factor,
    "dist3_factor": dist3_factor,
    "emp_factor": emp_factor,
    "short_trip_factor": short_trip_factor,
    "hwy_time": hwy_time,
    "walk_gencost": walk_gencost,
    "retail_ratio": retail_ratio,
    "ind_ratio": ind_ratio,
    "other_ratio": other_ratio,
    "log_file": log_file,
    "hbw_prods": hbw_prods,
    "target_attractions": target_attractions,
    "mask_external_dummys": mask_external_dummys,
    "district_match": district_match,
    "df_se_full": df_se_full,
    "pct_tel_hbw": pct_tel_hbw,
    "fac_tel_hbw": fac_tel_hbw,
    "UsedZones": used_zones,
    "df_obs_trips": df_obs_pivot,
    # --- Level 2 Groupings (Trip Attraction) ---
    "ret_ratio2": ret_ratio2,
    "ind_ratio2": ind_ratio2,
    "off_ratio2": off_ratio2,
    "oth_ratio2": oth_ratio2,
    # --- Level 3 Groupings (Compensation) ---
    "low_ratio3": low_ratio3,
    "mid_ratio3": mid_ratio3,
    "high_ratio3": high_ratio3,
    # parking
    "prk_temp": prk_temp,
    "prk_perm": prk_perm,
}


# Calibration


## Observed Targets


In [52]:
# read in observed
df_obs = static_data["df_obs_trips"]

# Convert 1-based TAZ numbers to 0-based Python indices
i_idx = df_obs["i"].astype(int).values - 1
j_idx = df_obs["j"].astype(int).values - 1

# Fetch the distance for every i-j pair directly from our loaded dist_mtx
df_obs["skim_dist"] = dist_mtx[i_idx, j_idx]

observed_avg_dist = {}

# Calculate the observed average distance for each segment
for seg in segments:  # e.g., '0veh_lo', '0veh_hi', etc.
    # Make sure the column name exactly matches the segment string in your loop
    total_obs_trips = df_obs[seg].sum()

    if total_obs_trips > 0:
        # Sum of (trips * distance) / total trips
        avg_dist = (df_obs[seg] * df_obs["skim_dist"]).sum() / total_obs_trips
        observed_avg_dist[seg] = avg_dist
    else:
        observed_avg_dist[seg] = 0.0
    print(f"  Target for {seg}: {observed_avg_dist[seg]:.2f} miles")


  Target for 0veh_lo: 4.14 miles
  Target for 0veh_hi: 4.64 miles
  Target for 1veh_lo: 9.00 miles
  Target for 1veh_hi: 9.18 miles
  Target for 2veh_lo: 7.05 miles
  Target for 2veh_hi: 11.35 miles


## Master Calibration Loop (Optional Distance Calibration)


In [ ]:
# --- CALIBRATION SETTINGS ---
calibrated_distance = (
    0  # 1 = Full Distance Calibration, 0 = Single Pass (Balancing Only)
)

print("\nInitializing Calibration...")

# Create a true, independent copy of the nested dictionary so we don't overwrite the original
current_coeffs = copy.deepcopy(initial_coeffs)

# calibration settings
max_calib_iterations = 20
learning_rate = 0.01  # How aggressively to adjust coefficients

for calib_iter in range(1, max_calib_iterations + 1):
    if calibrated_distance == 1:
        print(f"\n--- Starting Distance Calibration Iteration {calib_iter} ---")
    else:
        print("\n--- Running Destination Choice (Balancing Only) ---")

    # ------------------------------------------------------------------------
    # Recalculate Dynamic Variables (Size Terms depend on coefficients)
    # ------------------------------------------------------------------------
    size_terms = {
        seg: get_size_term(current_coeffs[seg], df_se_full) for seg in segments
    }
    adj_factors = np.zeros(used_zones)
    trips_data = {seg: {} for seg in segments}

    # ------------------------------------------------------------------------
    # Inner Iterative Balancing Loop
    # ------------------------------------------------------------------------
    for iterate in range(1, 11):
        total_trips_od = np.zeros((used_zones, used_zones))

        for seg in segments:
            c, st_vector = current_coeffs[seg], size_terms[seg]

            # -----------------------#
            # 1. Production alignment
            p_seg = hbw_prods[f"trips_{seg}"].values.copy()
            p_seg[mask_external_dummys[: len(p_seg)]] = 0

            ## -----------------------#
            ## 2. Utility calculation
            # utility = (
            #    (c["logsum"] * log_file[seg][:])
            #    + short_trip_factor
            #    + ((c["hwy_dist"] - c["distcal"]) * dist_mtx)
            #    + (dist2_factor * dist_sq)
            #    + (dist3_factor * dist_cu)
            #    + (c["hwy_time"] * hwy_time)
            #    + (c["transit_cost"] * walk_gencost)
            #    + (emp_factor * st_vector[None, :])
            #    + adj_factors[None, :]
            #    + (c["retail_ratio"] * retail_ratio)[None, :]
            #    + (c["industrial_ratio"] * ind_ratio)[None, :]
            #    + (c["other_ratio"] * other_ratio)[None, :]
            # )

            # -----------------------#
            # 2. Utility calculation (LEVEL 2 GROUPINGS)
            utility = (
                (c["logsum"] * log_file[seg][:])
                + short_trip_factor
                + ((c["hwy_dist"] - c["distcal"]) * dist_mtx)
                + (emp_factor * st_vector[None, :])
                + (c["prk_temp"] * prk_temp)[None, :]
                # + (c["prk_perm"] * prk_perm)[None, :]
                + adj_factors[None, :]
                # Level 2 SE Ratios (oth_ratio2 excluded as reference)
                + (c["ret_ratio2"] * ret_ratio2)[None, :]
                + (c["ind_ratio2"] * ind_ratio2)[None, :]
                + (c["off_ratio2"] * off_ratio2)[None, :]
            )

            ## -----------------------#
            ## 2. Utility calculation (LEVEL 3 GROUPINGS)
            # utility = (
            #    (c["logsum"] * log_file[seg][:])
            #    + short_trip_factor
            #    + ((c["hwy_dist"] - c["distcal"]) * dist_mtx)
            #    + (emp_factor * st_vector[None, :])
            #    + (c["prk_temp"] * prk_temp)[None, :]
            #    #+ (c["prk_perm"] * prk_perm)[None, :]
            #    + adj_factors[None, :]
            #    # Level 3 SE Ratios (low_ratio3 excluded as reference)
            #    + (c["mid_ratio3"] * mid_ratio3)[None, :]
            #    + (c["high_ratio3"] * high_ratio3)[None, :]
            # )

            np.fill_diagonal(utility, utility.diagonal() + c["intra"])
            utility += np.where(district_match, c["intradist"], 0)

            # -----------------------#
            # 3. Exponentiate
            exp_u = np.exp(utility)
            exp_u[p_seg <= 0, :] = 0
            exp_u[:, df_se_full["TOTEMP"].values <= 0] = 0
            exp_u[:, mask_external_dummys] = 0

            # -----------------------#
            # 4. Probabilities
            row_sums = exp_u.sum(axis=1)[:, np.newaxis]
            share_mtx = np.divide(
                exp_u, row_sums, out=np.zeros_like(exp_u), where=row_sums != 0
            )

            # -----------------------#
            # 5. Calculate Trips
            trips_mtx = share_mtx * p_seg[:, np.newaxis]
            trips_data[seg]["trips"] = trips_mtx
            total_trips_od += trips_mtx

        # -----------------------#
        # 6. Attraction balancing
        current_attractions = total_trips_od.sum(axis=0)
        adj_mask = (target_attractions > 0) & (~mask_external_dummys)
        new_step = np.zeros_like(adj_factors)
        safe_mask = adj_mask & (current_attractions > 0)
        new_step[safe_mask] = np.log(
            target_attractions[safe_mask] / current_attractions[safe_mask]
        )
        adj_factors += new_step

        # -----------------------#
        # 7. Convergence Check
        abs_diff = np.abs(current_attractions - target_attractions)
        pct_diff = np.zeros_like(abs_diff)
        np.divide(
            abs_diff, target_attractions, out=pct_diff, where=target_attractions > 0
        )
        percent_converged = np.sum(
            ((pct_diff < 0.02) | (abs_diff <= 10)) & (target_attractions > 0)
        ) / np.sum(target_attractions > 0)

        if percent_converged > 0.99:
            break

    # ------------------------------------------------------------------------
    # Export for Dashboard (Saves Iteration 1 or the Single Pass)
    # ------------------------------------------------------------------------
    if calib_iter == 1:
        print("Exporting results for dashboard...")
        export_run_for_dashboard(
            trips_data=trips_data,
            df_se_full=df_se_full,
            run_name="Test 0: Baseline (C33b)",
            file_name="test_0_baseline-33b",
        )

    # ------------------------------------------------------------------------
    # Distance Calibration Logic
    # ------------------------------------------------------------------------
    if calibrated_distance == 1:
        total_error = 0
        for seg in segments:
            simulated_trips = trips_data[seg]["trips"]
            total_seg_trips = simulated_trips.sum()

            simulated_avg_dist = (
                np.sum(simulated_trips * dist_mtx) / total_seg_trips
                if total_seg_trips > 0
                else 0
            )
            target_dist = observed_avg_dist[seg]
            error = simulated_avg_dist - target_dist
            total_error += abs(error)

            print(
                f"  {seg}: Sim Avg Dist = {simulated_avg_dist:.2f} | Target = {target_dist:.2f} | Error = {error:.2f}"
            )

            # Adjust the 'distcal' coefficient
            current_coeffs[seg]["distcal"] += error * learning_rate

        print(f"Total Absolute Error for Iteration {calib_iter}: {total_error:.2f}")

        if total_error < 0.05:
            print("\nCalibration targets reached! Breaking out of loop.")
            break
    else:
        print("Destination choice balancing complete. (Single Pass)")
        break

# ========================================================================
# FINAL EXPORT (ONLY IF CALIBRATION ACTUALLY RAN)
# ========================================================================
if calibrated_distance == 1 and calib_iter > 1:
    print("\nWriting FINAL calibrated coefficients and trips to file...")

    # 1. Export Coefficients (.block)
    out_coeff_path = path_outputs / "coefficients_cal8_adjusted2a.block"
    with open(out_coeff_path, "w") as f:
        f.write("; Calibrated HBW Destination Choice Coefficients\n")
        f.write(f"; Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        for seg in segments:
            f.write(f"; --- {seg} ---\n")
            for param, val in current_coeffs[seg].items():
                f.write(f"{param}_{seg} = {val:.6f}\n")
            f.write("\n")

    # 2. Export Final Trips (Dashboard)
    export_run_for_dashboard(
        trips_data=trips_data,
        df_se_full=df_se_full,
        run_name="Test 12: Adjusted2a (C28)",
        file_name="test_12_adjusted2a",
    )
    print(f"Final results saved! Coeffs at: {out_coeff_path}")



Initializing Calibration...

--- Running Destination Choice (Balancing Only) ---
Exporting results for dashboard...

  -> Exporting 'Test 0: Baseline (C33b)'...
  -> Dashboard CSV saved successfully to intermediate/model_runs/test_0_baseline-33b.csv!
Destination choice balancing complete. (Single Pass)


# Compare Results


In [25]:
run_files = [
    "intermediate/model_runs/test_0_baseline.csv",  # baseline c28 results with old coefs/constants
    "intermediate/model_runs/test_0_baseline-33b.csv",  # baseline c33b results with old coefs/constants
    "intermediate/model_runs/test_1_distance.csv",  # baseline c28 plus calibrated distance coefs
    "intermediate/model_runs/test_2_dist_bins.csv",  # baseline c28 plus calibrated distance coefs by bin
    "intermediate/model_runs/test_3_estimation1.csv",  # round 1 of estimated coefficients -- provided by andy
    "intermediate/model_runs/test_4_distance1.csv",  # round 1 of estimated coefficients plus calibrated distance coefs
    "intermediate/model_runs/test_4_distance1a.csv",  # round 1 of estimated coefficients plus calibrated distance and calibrated intrazonal coefs
    "intermediate/model_runs/test_5_estimation2.csv",  # round 2 of estimated coefficients (includes new param short_trip_factor)
    "intermediate/model_runs/test_6_distance2.csv",  # round 2 of estimated coefficients plus calibrated distance coeffs
    "intermediate/model_runs/test_7_estimation3.csv",  # round 3 of estimated coefficients (using trip weight this time)
    "intermediate/model_runs/test_8_distance3.csv",  # round 3 of estimated coefficients plus calibrated distance coeffs
    "intermediate/model_runs/test_9_adjusted1.csv",  # est3 + round 1 of adjusted coeffs (higher oth emp for lo inc, higher intradist for 1veh_lo and 2veh_lo)
    "intermediate/model_runs/test_10_adjusted1a.csv",  # est3 + round 1 of adjusted coeffs plus calibrated distance coeffs
    "intermediate/model_runs/test_11_adjusted2.csv",  # est3 + round 2 of adjusted coeffs (higher intradist for 1veh_lo and 2veh_lo)
    "intermediate/model_runs/test_12_adjusted2a.csv",  # est3 + round 2 of adjusted coeffs plus calibrated distance coeffs
]


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import os

# =========================================================
# 1. LOAD OBSERVED DATA
# =========================================================
obs_df = pd.read_csv("intermediate/obs_dist_hbw_sum.csv")
obs_df["veh_inc_clean"] = obs_df["veh_inc"].str.replace("HBW_", "")

# Force integers for safe merging
obs_df["p_DistLrg"] = obs_df["p_DistLrg"].astype(int)
obs_df["a_DistLrg"] = obs_df["a_DistLrg"].astype(int)

# =========================================================
# 2. LOAD YOUR MODEL RUNS
# =========================================================
# (Assuming run_files is defined in your environment)
mod_dfs = []
for file in run_files:
    if os.path.exists(file):
        mod_dfs.append(pd.read_csv(file))
    else:
        print(f"Notice: {file} not found yet. Skipping.")

if len(mod_dfs) > 0:
    mod_df = pd.concat(mod_dfs, ignore_index=True)
    mod_df["veh_inc_clean"] = mod_df["veh_inc"].str.replace("HBW_", "")
    mod_df["p_DistLrg"] = mod_df["p_DistLrg"].astype(int)
    mod_df["a_DistLrg"] = mod_df["a_DistLrg"].astype(int)
else:
    print("No model runs found! Go run your calibration loop first.")
    mod_df = pd.DataFrame(
        columns=["Source", "veh_inc_clean", "p_DistLrg", "a_DistLrg", "total_trips"]
    )


# =========================================================
# 3. BUILD THE BULLETPROOF DASHBOARD (Output Context)
# =========================================================
def show_dashboard(mod_data, obs_data):
    if mod_data.empty:
        print("No model data available to display.")
        return

    sources = mod_data["Source"].unique().tolist()
    veh_incs = sorted(obs_data["veh_inc_clean"].unique().tolist())

    source_dropdown = widgets.Dropdown(
        options=sources, value=sources[0], description="Model Run:"
    )

    # We use an Output widget to capture the plot rendering
    plot_output = widgets.Output()

    def update_plot(change):
        source = source_dropdown.value
        f_mod_all = mod_data[mod_data["Source"] == source]

        merged_all = pd.merge(
            obs_data,
            f_mod_all,
            on=["veh_inc_clean", "p_DistLrg", "a_DistLrg"],
            suffixes=("_obs", "_mod"),
        )
        merged_all = merged_all[merged_all["total_trips_obs"] > 0]

        # Calculate Global Metrics
        if not merged_all.empty:
            total_abs_error = np.abs(
                merged_all["total_trips_mod"] - merged_all["total_trips_obs"]
            ).sum()
            total_obs = merged_all["total_trips_obs"].sum()
            wmape = (total_abs_error / total_obs) * 100 if total_obs > 0 else 0

            segment_slopes = []
            for seg in veh_incs:
                seg_df = merged_all[merged_all["veh_inc_clean"] == seg]
                if len(seg_df) > 1:
                    x_s = seg_df["total_trips_obs"]
                    y_s = seg_df["total_trips_mod"]
                    slope = (
                        np.sum(x_s * y_s) / np.sum(x_s**2) if np.sum(x_s**2) > 0 else 0
                    )
                    segment_slopes.append(slope)

            avg_slope = np.mean(segment_slopes) if segment_slopes else 0
            main_title = f"Run: {source} | WMAPE: {wmape:.1f}% | Avg Segment Slope: {avg_slope:.2f} (Target 1.0)"
        else:
            main_title = f"Run: {source} | Error: No Data Found"

        # Construct a FRESH Figure to avoid any JS state sync errors
        fig = make_subplots(
            rows=2,
            cols=3,
            subplot_titles=[f"Segment: {seg}" for seg in veh_incs],
            horizontal_spacing=0.08,
            vertical_spacing=0.15,
        )

        all_shapes = []

        for i, seg in enumerate(veh_incs):
            row = (i // 3) + 1
            col = (i % 3) + 1
            axis_num = i + 1
            ax_pfx = "" if axis_num == 1 else str(axis_num)

            f_mod = f_mod_all[f_mod_all["veh_inc_clean"] == seg]
            f_obs = obs_data[obs_data["veh_inc_clean"] == seg]

            merged = pd.merge(
                f_obs,
                f_mod,
                on=["p_DistLrg", "a_DistLrg"],
                suffixes=("_obs", "_mod"),
            )
            merged = merged[merged["total_trips_obs"] > 0]

            if merged.empty:
                fig.layout.annotations[i].text = f"{seg} (No Data)"
                continue

            x_vals = merged["total_trips_obs"]
            y_vals = merged["total_trips_mod"]
            max_val = max(x_vals.max(), y_vals.max())

            hover_text = [
                f"Dist: {p} to {a}<br>Obs: {o:.0f}<br>Mod: {m:.0f}"
                for p, a, o, m in zip(
                    merged["p_DistLrg"], merged["a_DistLrg"], x_vals, y_vals
                )
            ]

            # 1. Add Scatter
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=y_vals,
                    mode="markers",
                    marker=dict(color="royalblue", opacity=0.6),
                    hovertext=hover_text,
                    name=seg,
                ),
                row=row,
                col=col,
            )

            # 2. Add Trendline
            if len(merged) > 1:
                slope = np.sum(x_vals * y_vals) / np.sum(x_vals**2)
                fig.add_trace(
                    go.Scatter(
                        x=[0, max_val],
                        y=[0, slope * max_val],
                        mode="lines",
                        line=dict(color="royalblue", dash="dash", width=2),
                        name=f"Trend {seg}",
                        hoverinfo="skip",
                    ),
                    row=row,
                    col=col,
                )
                fig.layout.annotations[i].text = f"{seg} | Slope: {slope:.2f}"

            # 3. Set Axis Ranges
            fig.layout[f"xaxis{ax_pfx}"].range = [0, max_val * 1.1]
            fig.layout[f"yaxis{ax_pfx}"].range = [0, max_val * 1.1]
            fig.layout[f"xaxis{ax_pfx}"].title = "Observed Trips"
            fig.layout[f"yaxis{ax_pfx}"].title = "Modeled Trips"

            # 4. Build Fan Lines
            xref = f"x{ax_pfx}" if ax_pfx else "x"
            yref = f"y{ax_pfx}" if ax_pfx else "y"
            for r in [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]:
                all_shapes.append(
                    dict(
                        type="line",
                        x0=0,
                        y0=0,
                        x1=max_val,
                        y1=max_val * r,
                        xref=xref,
                        yref=yref,
                        layer="below",
                        line=dict(
                            color="gray",
                            width=2 if r == 1.0 else 1,
                            dash="dot" if r != 1.0 else "solid",
                        ),
                    )
                )

        # Finalize Figure Layout
        fig.update_layout(
            height=800,
            width=1200,
            showlegend=False,
            template="plotly_white",
            title_text=main_title,
            shapes=all_shapes,
        )

        # Clear the old plot and render the new one
        with plot_output:
            plot_output.clear_output(wait=True)
            display(fig)

    # Listen for dropdown changes
    source_dropdown.observe(update_plot, names="value")

    # Run once to initialize
    update_plot(None)

    # Display UI
    display(widgets.VBox([source_dropdown, plot_output]))


# Call the dashboard
show_dashboard(mod_df, obs_df)


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import os


# =========================================================
# 0. HELPER FUNCTION TO TAG DIMENSIONS
# =========================================================
def tag_dimensions(df):
    """
    Parses the 6 combined 'veh_inc_clean' categories into distinct
    'Vehicle' and 'Income' columns so we can group by them dynamically.
    """
    df = df.copy()

    # ---------------------------------------------------------
    # USER ALERT: Update these rules based on your exact strings!
    # ---------------------------------------------------------
    def get_veh(val):
        val = str(val).lower()
        # Look for 0, 1, or 2 in the string to determine vehicle class
        if "0" in val:
            return "0 Veh"
        elif "1" in val:
            return "1 Veh"
        else:
            return "2+ Veh"  # Captures '2' or anything else

    def get_inc(val):
        val = str(val).lower()
        # Look for keywords indicating low vs high income
        if "lo" in val or "inc1" in val:
            return "Low Income"
        else:
            return "High Income"

    df["Vehicle"] = df["veh_inc_clean"].apply(get_veh)
    df["Income"] = df["veh_inc_clean"].apply(get_inc)

    return df


# =========================================================
# 1. LOAD & PREP OBSERVED DATA
# =========================================================
obs_df = pd.read_csv("intermediate/obs_dist_hbw_sum.csv")
obs_df["veh_inc_clean"] = obs_df["veh_inc"].str.replace("HBW_", "")
obs_df["p_DistLrg"] = obs_df["p_DistLrg"].astype(int)
obs_df["a_DistLrg"] = obs_df["a_DistLrg"].astype(int)

# Apply our tagging function
obs_df = tag_dimensions(obs_df)

# =========================================================
# 2. LOAD & PREP YOUR MODEL RUNS
# =========================================================
mod_dfs = []
for file in run_files:
    if os.path.exists(file):
        mod_dfs.append(pd.read_csv(file))
    else:
        print(f"Notice: {file} not found yet. Skipping.")

if len(mod_dfs) > 0:
    mod_df = pd.concat(mod_dfs, ignore_index=True)
    mod_df["veh_inc_clean"] = mod_df["veh_inc"].str.replace("HBW_", "")
    mod_df["p_DistLrg"] = mod_df["p_DistLrg"].astype(int)
    mod_df["a_DistLrg"] = mod_df["a_DistLrg"].astype(int)

    # Apply our tagging function
    mod_df = tag_dimensions(mod_df)
else:
    print("No model runs found! Go run your calibration loop first.")
    mod_df = pd.DataFrame(
        columns=[
            "Source",
            "veh_inc_clean",
            "Vehicle",
            "Income",
            "p_DistLrg",
            "a_DistLrg",
            "total_trips",
        ]
    )


# =========================================================
# 3. BUILD THE DYNAMIC DASHBOARD
# =========================================================
def show_dynamic_dashboard(mod_data, obs_data):
    if mod_data.empty:
        print("No model data available to display.")
        return

    sources = mod_data["Source"].unique().tolist()

    # Create Widgets
    source_dropdown = widgets.Dropdown(
        options=sources, value=sources[0], description="Model Run:"
    )

    # Using a ToggleButton for a cleaner UI experience between the two modes
    dimension_toggle = widgets.ToggleButtons(
        options=["Income", "Vehicle"], description="View By:", button_style="info"
    )

    plot_output = widgets.Output()

    def update_plot(*args):
        source = source_dropdown.value
        dim = dimension_toggle.value  # Will be 'Income' or 'Vehicle'

        f_mod_all = mod_data[mod_data["Source"] == source]

        # ---------------------------------------------------------
        # DYNAMIC AGGREGATION
        # We group by the selected dimension and sum the trips.
        # This effectively squashes the 6 categories down to 2 or 3.
        # ---------------------------------------------------------
        obs_agg = obs_data.groupby([dim, "p_DistLrg", "a_DistLrg"], as_index=False)[
            "total_trips"
        ].sum()
        mod_agg = f_mod_all.groupby([dim, "p_DistLrg", "a_DistLrg"], as_index=False)[
            "total_trips"
        ].sum()

        merged_all = pd.merge(
            obs_agg,
            mod_agg,
            on=[dim, "p_DistLrg", "a_DistLrg"],
            suffixes=("_obs", "_mod"),
        )
        merged_all = merged_all[merged_all["total_trips_obs"] > 0]

        # Determine segments to plot based on the toggle
        if dim == "Income":
            segments = ["Low Income", "High Income"]
            cols = 2
        else:
            segments = ["0 Veh", "1 Veh", "2+ Veh"]
            cols = 3

        # Calculate Global Metrics
        if not merged_all.empty:
            total_abs_error = np.abs(
                merged_all["total_trips_mod"] - merged_all["total_trips_obs"]
            ).sum()
            total_obs = merged_all["total_trips_obs"].sum()
            wmape = (total_abs_error / total_obs) * 100 if total_obs > 0 else 0

            segment_slopes = []
            for seg in segments:
                seg_df = merged_all[merged_all[dim] == seg]
                if len(seg_df) > 1:
                    x_s = seg_df["total_trips_obs"]
                    y_s = seg_df["total_trips_mod"]
                    slope = (
                        np.sum(x_s * y_s) / np.sum(x_s**2) if np.sum(x_s**2) > 0 else 0
                    )
                    segment_slopes.append(slope)

            avg_slope = np.mean(segment_slopes) if segment_slopes else 0
            main_title = f"Run: {source} | Aggregated By: {dim} | WMAPE: {wmape:.1f}% | Avg Slope: {avg_slope:.2f}"
        else:
            main_title = f"Run: {source} | Error: No Data Found"

        # Rebuild the Figure Grid dynamically (1 row, 2 or 3 columns)
        fig = make_subplots(
            rows=1,
            cols=cols,
            subplot_titles=[f"{seg}" for seg in segments],
            horizontal_spacing=0.08,
        )

        all_shapes = []

        for i, seg in enumerate(segments):
            col = i + 1
            ax_pfx = "" if col == 1 else str(col)

            merged = merged_all[merged_all[dim] == seg]

            if merged.empty:
                fig.layout.annotations[i].text = f"{seg} (No Data)"
                continue

            x_vals = merged["total_trips_obs"]
            y_vals = merged["total_trips_mod"]
            max_val = max(x_vals.max(), y_vals.max())

            hover_text = [
                f"Dist: {p} to {a}<br>Obs: {o:.0f}<br>Mod: {m:.0f}"
                for p, a, o, m in zip(
                    merged["p_DistLrg"], merged["a_DistLrg"], x_vals, y_vals
                )
            ]

            # 1. Add Scatter
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=y_vals,
                    mode="markers",
                    marker=dict(color="royalblue", opacity=0.6),
                    hovertext=hover_text,
                    name=seg,
                ),
                row=1,
                col=col,
            )

            # 2. Add Trendline
            if len(merged) > 1:
                slope = np.sum(x_vals * y_vals) / np.sum(x_vals**2)
                fig.add_trace(
                    go.Scatter(
                        x=[0, max_val],
                        y=[0, slope * max_val],
                        mode="lines",
                        line=dict(color="royalblue", dash="dash", width=2),
                        name=f"Trend {seg}",
                        hoverinfo="skip",
                    ),
                    row=1,
                    col=col,
                )
                fig.layout.annotations[i].text = f"{seg} | Slope: {slope:.2f}"

            # 3. Set Axis Ranges dynamically based on this specific subplot
            fig.layout[f"xaxis{ax_pfx}"].range = [0, max_val * 1.05]
            fig.layout[f"yaxis{ax_pfx}"].range = [0, max_val * 1.05]
            fig.layout[f"xaxis{ax_pfx}"].title = "Observed Trips"
            if col == 1:
                fig.layout[f"yaxis{ax_pfx}"].title = "Modeled Trips"

            # 4. Build Fan Lines
            xref = f"x{ax_pfx}" if ax_pfx else "x"
            yref = f"y{ax_pfx}" if ax_pfx else "y"
            for r in [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]:
                all_shapes.append(
                    dict(
                        type="line",
                        x0=0,
                        y0=0,
                        x1=max_val,
                        y1=max_val * r,
                        xref=xref,
                        yref=yref,
                        layer="below",
                        line=dict(
                            color="gray",
                            width=2 if r == 1.0 else 1,
                            dash="dot" if r != 1.0 else "solid",
                        ),
                    )
                )

        # Finalize Figure Layout
        fig.update_layout(
            height=500,
            width=1200,  # Made slightly shorter since it's only 1 row now
            showlegend=False,
            template="plotly_white",
            title_text=main_title,
            shapes=all_shapes,
        )

        # Clear the old plot and render the new one seamlessly
        with plot_output:
            plot_output.clear_output(wait=True)
            display(fig)

    # Listen for changes on BOTH widgets
    source_dropdown.observe(update_plot, names="value")
    dimension_toggle.observe(update_plot, names="value")

    # Run once to initialize
    update_plot()

    # Display UI - Arranging controls horizontally on top
    controls = widgets.HBox([source_dropdown, dimension_toggle])
    display(widgets.VBox([controls, plot_output]))


# Call the dashboard
show_dynamic_dashboard(mod_df, obs_df)
